# SEN2SR Lite → Super-resolution → Spectral Index Comparison

This notebook reads the Copernicus ZIP from Google Drive, extracts it automatically, identifies the Sentinel-2 bands, prepares a 10-band stack, runs pretrained SEN2SR Lite, and displays RGB, NDVI, NDWI and NDBI comparisons.

## Bands required by this notebook

The 10-band input order is:

`B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12`

Use Sentinel-2 L2A analytical data, not a JPEG/PNG visual export. Start with a small study-area download. The model prediction is not ground truth, and this notebook does not fine-tune the model or implement a spectral-index loss.

## Cell 1 — Install packages

Run this cell first in Google Colab.

In [ ]:
%pip install -q sen2sr mlstac rasterio matplotlib numpy

## Cell 2 — Imports and GPU check

In [ ]:
import os, re, zipfile, shutil
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from affine import Affine
import torch
import mlstac

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. In Colab, choose Runtime → Change runtime type → T4 GPU if available.")

Device: cuda
GPU: Tesla T4


## Cell 3 — Connect Drive and extract `Browser_images.zip`

Keep the ZIP in `My Drive/Datasets/`. This cell extracts it automatically into Colab's temporary `/content` storage. You do not need to unzip it manually.

In [29]:
from google.colab import drive
drive.mount("/content/drive")

# Change this only if you put the ZIP in a different folder or used a different filename.
ZIP_PATH = "/content/drive/MyDrive/Project_Folder/Datasets/Browser_images_1.zip"

# Google Drive sometimes displays the name without the .zip extension.
if not os.path.exists(ZIP_PATH):
    alternative = os.path.join(DRIVE_DATASET_DIR, "Browser_images")
    if os.path.exists(alternative):
        ZIP_PATH = alternative

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(
        f"ZIP not found at: {ZIP_PATH}\n"
        "Check that Browser_images.zip is directly inside My Drive/Datasets."
    )

EXTRACT_DIR = "/content/copernicus_data"
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

if not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError(f"The selected file is not a valid ZIP archive: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(EXTRACT_DIR)

print("ZIP extracted from:", ZIP_PATH)
print("Extracted to:", EXTRACT_DIR)
print("\nRaster files found:")
raster_files = []
for root, dirs, filenames in os.walk(EXTRACT_DIR):
    for filename in filenames:
        if filename.lower().endswith((".tif", ".tiff", ".jp2")):
            path = os.path.join(root, filename)
            raster_files.append(path)
            print(path)

if not raster_files:
    raise FileNotFoundError(
        "No .tif, .tiff, or .jp2 files found inside the ZIP. "
        "The archive may contain a different format or a nested download that needs inspection."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ZIP extracted from: /content/drive/MyDrive/Project_Folder/Datasets/Browser_images_1.zip
Extracted to: /content/copernicus_data

Raster files found:
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B04_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B03_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B07_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B08_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B12_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B11_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_B06_(Raw).tiff
/content/copernicus_data/2026-03-04-00_00_2026-03-04-23_59_Sentinel-2_L2A_Highlight_Optimized_Natura

## Cell 4 — Identify bands and build a 10-band GeoTIFF

This cell supports either:
- a single 10-band TIFF already ordered correctly, or
- separate band files named with standard tokens such as `..._B02_10m.jp2`.

If several files for the same band/resolution are present, the notebook stops rather than guessing. All bands must come from the same acquisition/granule. It aligns individual bands to the B02 grid.

In [30]:
BAND_NAMES = [
    "B02", "B03", "B04", "B05", "B06",
    "B07", "B08", "B8A", "B11", "B12"
]

# Case A: one multiband TIFF containing all ten bands.
multiband_candidates = []
for path in raster_files:
    if path.lower().endswith((".tif", ".tiff")):
        try:
            with rasterio.open(path) as src:
                if src.count == 10:
                    multiband_candidates.append(path)
        except Exception:
            pass

if len(multiband_candidates) == 1:
    INPUT_TIF = multiband_candidates[0]
    print("Using 10-band TIFF:", INPUT_TIF)
elif len(multiband_candidates) > 1:
    raise ValueError(
        "Found multiple 10-band TIFFs. Please keep only one intended input TIFF in the ZIP "
        "or set INPUT_TIF manually to the correct file."
    )
else:
    # Case B: separate band files. Match band names from standard SAFE/browser filenames.
    found = {}
    duplicates = {}
    for path in raster_files:
        upper = os.path.basename(path).upper()
        matches = [b for b in BAND_NAMES if re.search(rf"(^|[_\.-]){b}([_\.-]|$)", upper)]
        if not matches:
            continue
        b = matches[0]
        if b in found:
            duplicates.setdefault(b, [found[b]]).append(path)
        else:
            found[b] = path

    missing = [b for b in BAND_NAMES if b not in found]
    if duplicates:
        raise ValueError(
            "Multiple files found for these bands: " + str(duplicates) +
            ". The notebook will not guess which resolution/granule to use. "
            "Use one file per band from the same granule, or create a correctly ordered 10-band TIFF."
        )
    if missing:
        print("Band files identified:", found)
        raise ValueError(
            "Could not identify all ten required bands. Missing: " + ", ".join(missing) +
            ". The ZIP may contain a visualized export or separate TIFFs whose filenames do not identify raw band names. "
            "See the printed raster file list and we can adapt the loader."
        )

    # B02 is the reference 10 m grid.
    with rasterio.open(found["B02"]) as ref:
        ref_profile = ref.profile.copy()
        ref_transform, ref_crs = ref.transform, ref.crs
        height, width = ref.height, ref.width

    if ref_crs is None:
        raise ValueError("B02 file has no CRS. Use georeferenced raw Copernicus band files.")

    stack = []
    for b in BAND_NAMES:
        with rasterio.open(found[b]) as src:
            if src.crs is None:
                raise ValueError(f"{b} has no CRS metadata.")
            dest = np.zeros((height, width), dtype=np.float32)
            reproject(
                source=rasterio.band(src, 1),
                destination=dest,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                resampling=Resampling.bilinear
            )
            stack.append(dest)

    stack = np.stack(stack).astype(np.float32)
    INPUT_TIF = "/content/sentinel2_10band_stack.tif"
    profile = ref_profile.copy()
    profile.update(driver="GTiff", count=10, height=height, width=width,
                   dtype="float32", compress="deflate", nodata=None)
    with rasterio.open(INPUT_TIF, "w", **profile) as dst:
        dst.write(stack)
    print("Created stack:", INPUT_TIF)

print("\nInput TIFF:", INPUT_TIF)
with rasterio.open(INPUT_TIF) as src:
    print("Bands:", src.count, "| shape:", (src.height, src.width), "| dtype:", src.dtypes)
    print("CRS:", src.crs, "| resolution:", src.res)

Created stack: /content/sentinel2_10band_stack.tif

Input TIFF: /content/sentinel2_10band_stack.tif
Bands: 10 | shape: (135, 236) | dtype: ('float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32', 'float32')
CRS: EPSG:4326 | resolution: (9.001271186439978e-05, 7.785925925926702e-05)


## Cell 5 — Load input and set reflectance scaling

Copernicus Browser analytical exports may use different numeric scaling than original SAFE product values. Inspect the printed range. For common Sentinel-2 SAFE integer surface-reflectance data, values are often scaled by 10000; a browser export may instead be scaled to its selected output bit depth.

**Do not blindly divide by 10000 if the values are already scaled differently.** This cell uses a simple range-based initial guess, but verify it before scientific use.

In [31]:
with rasterio.open(INPUT_TIF) as src:
    raw = src.read().astype(np.float32)
    input_profile = src.profile.copy()
    input_transform = src.transform
    input_crs = src.crs
    nodata = src.nodata

if raw.shape[0] != 10:
    raise ValueError(f"Expected 10 bands, got {raw.shape[0]}.")

finite = raw[np.isfinite(raw)]
print("Raw pixel range:", float(np.nanpercentile(finite, 1)), "to", float(np.nanpercentile(finite, 99)))
print("dtype:", raw.dtype, "| nodata:", nodata)

# EDIT THIS after inspecting the range above:
# - use 10000.0 for common SAFE integer reflectance values scaled by 10000
# - use 1.0 if already stored as reflectance in the 0–1 range
# - for Copernicus Browser 16-bit rendered analytical exports, inspect the export scaling/documentation first.
SCALE_FACTOR = 10000.0

X_np = raw / SCALE_FACTOR
if nodata is not None:
    X_np[raw == nodata] = np.nan
print("Scaled reflectance range (1st–99th percentile):",
      float(np.nanpercentile(X_np, 1)), "to", float(np.nanpercentile(X_np, 99)))

# Fill invalid pixels only to prevent model crashes; cloud/NoData masking is still needed for valid science.
X_np = np.nan_to_num(X_np, nan=0.0, posinf=0.0, neginf=0.0)
X = torch.from_numpy(X_np).float().to(device)
print("Model input shape:", tuple(X.shape))

Raw pixel range: 2077.0 to 20152.0
dtype: float32 | nodata: None
Scaled reflectance range (1st–99th percentile): 0.2076999992132187 to 2.015199899673462
Model input shape: (10, 135, 236)


## Cell 6 — Download and load pretrained SEN2SR Lite

This uses the repository's 10-band `SEN2SRLite` model entry. The code is for inference, not fine-tuning.

In [ ]:
MODEL_DIR = "/content/model/SEN2SRLite"
os.makedirs(MODEL_DIR, exist_ok=True)

mlstac.download(
    file="https://huggingface.co/tacofoundation/sen2sr/resolve/main/SEN2SRLite/main/mlm.json",
    output_dir=MODEL_DIR,
)
model = mlstac.load(MODEL_DIR).compiled_model(device=device)
model = model.to(device)
model.eval()
print("SEN2SR Lite loaded.")

SEN2SR Lite loaded.


## Cell 7 — Run model inference

In [32]:
import torch
import torch.nn.functional as F

# X shape: (10, H, W)
X = X.float().to(device)
_, H, W = X.shape

PATCH = 128
OVERLAP = 32
STRIDE = PATCH - OVERLAP
SCALE = 4

# Pad image to ensure every patch is 128x128
pad_h = max(0, PATCH - H)
pad_w = max(0, PATCH - W)

X_pad = F.pad(X, (0, pad_w, 0, pad_h), mode="replicate")
Hp, Wp = X_pad.shape[-2:]

# Accumulate overlapping predictions
out = torch.zeros(
    (10, Hp * SCALE, Wp * SCALE),
    device=device,
    dtype=X.dtype
)
weights = torch.zeros(
    (1, Hp * SCALE, Wp * SCALE),
    device=device,
    dtype=X.dtype
)

with torch.inference_mode():
    for y in range(0, Hp, STRIDE):
        for x in range(0, Wp, STRIDE):
            # Ensure each patch stays inside the padded image
            y0 = min(y, Hp - PATCH)
            x0 = min(x, Wp - PATCH)

            patch = X_pad[:, y0:y0+PATCH, x0:x0+PATCH].unsqueeze(0)
            pred = model(patch).squeeze(0)

            oy, ox = y0 * SCALE, x0 * SCALE
            out[:, oy:oy+PATCH*SCALE, ox:ox+PATCH*SCALE] += pred
            weights[:, oy:oy+PATCH*SCALE, ox:ox+PATCH*SCALE] += 1

superX = out / weights.clamp_min(1)

# Remove padding from output
superX = superX[:, :H*SCALE, :W*SCALE]

print("Input shape:", tuple(X.shape))
print("Super-resolved output:", tuple(superX.shape))
print("Inference completed.")

Input shape: (10, 135, 236)
Super-resolved output: (10, 540, 944)
Inference completed.


## Cell 8 — Calculate NDVI, NDWI and NDBI

- NDVI = (B08 − B04) / (B08 + B04)
- NDWI (Green–NIR) = (B03 − B08) / (B03 + B08)
- NDBI = (B11 − B08) / (B11 + B08)

These indices are only as meaningful as the band alignment, scaling, masking and output-band contract.

In [33]:
import torch

# Band indices in the current input order:
# B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12
B03, B04, B08, B11 = 1, 2, 6, 8

def normalized_difference(a, b, eps=1e-8):
    return (a - b) / (a + b + eps)

# Input indices
ndvi_in = normalized_difference(X[B08], X[B04])
ndwi_in = normalized_difference(X[B03], X[B08])
ndbi_in = normalized_difference(X[B11], X[B08])

# Super-resolved indices
ndvi_out = normalized_difference(superX[B08], superX[B04])
ndwi_out = normalized_difference(superX[B03], superX[B08])
ndbi_out = normalized_difference(superX[B11], superX[B08])

print("NDVI input/output:", ndvi_in.shape, ndvi_out.shape)
print("NDWI input/output:", ndwi_in.shape, ndwi_out.shape)
print("NDBI input/output:", ndbi_in.shape, ndbi_out.shape)

NDVI input/output: torch.Size([135, 236]) torch.Size([540, 944])
NDWI input/output: torch.Size([135, 236]) torch.Size([540, 944])
NDBI input/output: torch.Size([135, 236]) torch.Size([540, 944])


## Cell 9 — Display original RGB, predicted RGB, and all index maps

In [34]:

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import numpy as np

# Band order: B02, B03, B04, B05, B06,
#             B07, B08, B8A, B11, B12
# RGB order: B04, B03, B02

target = superX.shape[-2:]

# Upscale original input to match SEN2SR output dimensions
X_upscaled = F.interpolate(
    X.unsqueeze(0).float(),
    size=target,
    mode="bilinear",
    align_corners=False
).squeeze(0)

def to_numpy(img):
    if isinstance(img, torch.Tensor):
        return img.detach().float().cpu().numpy()
    return np.asarray(img)

def rgb_array(img):
    # B04, B03, B02
    rgb = to_numpy(img[[2, 1, 0]])
    return np.moveaxis(rgb, 0, -1)

def shared_stretch(before, after, low=2, high=98):
    # Use common limits so before/after brightness is comparable
    combined = np.concatenate([
        before[np.isfinite(before)].ravel(),
        after[np.isfinite(after)].ravel()
    ])

    if combined.size == 0:
        return np.zeros_like(before), np.zeros_like(after)

    lo, hi = np.percentile(combined, [low, high])
    if hi <= lo:
        hi = lo + 1e-8

    return (
        np.clip((before - lo) / (hi - lo), 0, 1),
        np.clip((after - lo) / (hi - lo), 0, 1)
    )

def resize_index(img, target):
    # Resize an index map to the output dimensions
    if isinstance(img, np.ndarray):
        img = torch.from_numpy(img)
    img = img.detach().float().cpu()

    if img.ndim == 3:
        img = img.squeeze()

    return F.interpolate(
        img[None, None],
        size=target,
        mode="bilinear",
        align_corners=False
    )[0, 0].numpy()

# -------------------------------
# 1. Overall multispectral display
# -------------------------------
# Average across all 10 bands at each pixel.
# Useful for a broad diagnostic view, not a standard satellite composite.
overall_before = to_numpy(X_upscaled).mean(axis=0)
overall_after = to_numpy(superX).mean(axis=0)

overall_b, overall_a = shared_stretch(
    overall_before, overall_after
)

# -------------------------------
# 2. RGB display
# -------------------------------
rgb_before_raw = rgb_array(X_upscaled)
rgb_after_raw = rgb_array(superX)

# Shared limits per channel, computed across both images
rgb_before = np.zeros_like(rgb_before_raw, dtype=np.float32)
rgb_after = np.zeros_like(rgb_after_raw, dtype=np.float32)

for c in range(3):
    b, a = shared_stretch(
        rgb_before_raw[..., c],
        rgb_after_raw[..., c]
    )
    rgb_before[..., c] = b
    rgb_after[..., c] = a

# -------------------------------
# 3. Index maps
# -------------------------------
ndvi_before = resize_index(ndvi_in, target)
ndwi_before = resize_index(ndwi_in, target)
ndbi_before = resize_index(ndbi_in, target)

ndvi_after = to_numpy(ndvi_out)
ndwi_after = to_numpy(ndwi_out)
ndbi_after = to_numpy(ndbi_out)

# -------------------------------
# 4. Plot before and after
# -------------------------------
fig, axes = plt.subplots(5, 2, figsize=(14, 25))

# Overall
axes[0, 0].imshow(overall_b, cmap="gray", vmin=0, vmax=1)
axes[0, 0].set_title("Overall Multispectral — Before (Upscaled)")

axes[0, 1].imshow(overall_a, cmap="gray", vmin=0, vmax=1)
axes[0, 1].set_title("Overall Multispectral — After (SEN2SR)")

# RGB
axes[1, 0].imshow(rgb_before)
axes[1, 0].set_title("RGB — Before (Upscaled)")

axes[1, 1].imshow(rgb_after)
axes[1, 1].set_title("RGB — After (SEN2SR)")

# NDVI
axes[2, 0].imshow(ndvi_before, cmap="RdYlGn", vmin=-1, vmax=1)
axes[2, 0].set_title("NDVI — Before (Resized)")

axes[2, 1].imshow(ndvi_after, cmap="RdYlGn", vmin=-1, vmax=1)
axes[2, 1].set_title("NDVI — After (SEN2SR)")

# NDWI
axes[3, 0].imshow(ndwi_before, cmap="BrBG", vmin=-1, vmax=1)
axes[3, 0].set_title("NDWI — Before (Resized)")

axes[3, 1].imshow(ndwi_after, cmap="BrBG", vmin=-1, vmax=1)
axes[3, 1].set_title("NDWI — After (SEN2SR)")

# NDBI
axes[4, 0].imshow(ndbi_before, cmap="coolwarm", vmin=-1, vmax=1)
axes[4, 0].set_title("NDBI — Before (Resized)")

axes[4, 1].imshow(ndbi_after, cmap="coolwarm", vmin=-1, vmax=1)
axes[4, 1].set_title("NDBI — After (SEN2SR)")

for ax in axes.flat:
    ax.axis("off")

plt.tight_layout()
plt.show()


Output hidden; open in https://colab.research.google.com to view.

## Cell 10 — Difference maps and diagnostic statistics

In [35]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Move tensors to CPU for plotting
inp = X.detach().float().cpu()
sr = superX.detach().float().cpu()

# Resize input to output size only for spatial comparison
inp_up = F.interpolate(
    inp.unsqueeze(0),
    size=sr.shape[-2:],
    mode="bilinear",
    align_corners=False
).squeeze(0)

def rgb_view(img):
    # RGB = B04, B03, B02
    rgb = img[[2, 1, 0]].permute(1, 2, 0).numpy()
    lo, hi = np.percentile(rgb, [2, 98])
    return np.clip((rgb - lo) / max(hi - lo, 1e-8), 0, 1)

# Plot the same crop from each image
H, W = sr.shape[-2:]
y0, y1 = H // 3, 2 * H // 3
x0, x1 = W // 3, 2 * W // 3

fig, ax = plt.subplots(2, 2, figsize=(14, 12))

ax[0, 0].imshow(rgb_view(inp_up))
ax[0, 0].set_title("Input resized to output grid")

ax[0, 1].imshow(rgb_view(sr))
ax[0, 1].set_title("SEN2SR ×4 RGB")

ax[1, 0].imshow(rgb_view(inp_up)[y0:y1, x0:x1])
ax[1, 0].set_title("Input — zoomed crop")

ax[1, 1].imshow(rgb_view(sr)[y0:y1, x0:x1])
ax[1, 1].set_title("SEN2SR — same crop")

for a in ax.flat:
    a.axis("off")

plt.tight_layout()
plt.show()

# Index comparison and difference maps
def safe_index(a, b, eps=1e-6):
    den = a + b
    result = torch.full_like(den, float("nan"))
    valid = den.abs() > eps
    result[valid] = (a[valid] - b[valid]) / den[valid]
    return result

# B02, B03, B04, B05, B06, B07, B08, B8A, B11, B12
index_specs = [
    ("NDVI", 6, 2),
    ("NDWI", 1, 6),
    ("NDBI", 8, 6),
]

fig, axes = plt.subplots(3, 3, figsize=(16, 14))

for row, (name, band_a, band_b) in enumerate(index_specs):
    before = safe_index(inp_up[band_a], inp_up[band_b]).numpy()
    after = safe_index(sr[band_a], sr[band_b]).numpy()

    difference = after - before
    valid = np.isfinite(difference)

    if valid.any():
        # Robust limits improve visibility without changing values
        limit = max(np.percentile(np.abs(difference[valid]), 98), 1e-6)

        axes[row, 0].imshow(before, cmap="viridis", vmin=-1, vmax=1)
        axes[row, 0].set_title(f"{name} — resized input")

        axes[row, 1].imshow(after, cmap="viridis", vmin=-1, vmax=1)
        axes[row, 1].set_title(f"{name} — SEN2SR")

        im = axes[row, 2].imshow(
            difference, cmap="RdBu_r",
            vmin=-limit, vmax=limit
        )
        axes[row, 2].set_title(
            f"{name} — difference\n±{limit:.4f} display scale"
        )
        fig.colorbar(im, ax=axes[row, 2], shrink=0.75)

    for a in axes[row]:
        a.axis("off")

plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [28]:

import numpy as np
import pandas as pd
import torch

# --------------------------------------------------
# 1. Calculate indices safely from the 10-band image
# Band order: B02, B03, B04, B05, B06,
#             B07, B08, B8A, B11, B12
# --------------------------------------------------

def calculate_index(img, band_a, band_b, eps=1e-6):
    """
    Index = (A - B) / (A + B)
    band_a and band_b are zero-based band indices.
    """
    a = img[band_a].float()
    b = img[band_b].float()
    denominator = a + b

    valid = (
        torch.isfinite(a)
        & torch.isfinite(b)
        & torch.isfinite(denominator)
        & (denominator.abs() > eps)
    )

    result = torch.full_like(denominator, float("nan"))
    result[valid] = (
        (a[valid] - b[valid]) / denominator[valid]
    )

    return result


# Compute indices directly from the images being compared
# NDVI = (B08 - B04) / (B08 + B04)
# NDWI = (B03 - B08) / (B03 + B08)
# NDBI = (B11 - B08) / (B11 + B08)

indices_before = {
    "NDVI": calculate_index(X_upscaled, 6, 2),
    "NDWI": calculate_index(X_upscaled, 1, 6),
    "NDBI": calculate_index(X_upscaled, 8, 6),
}

indices_after = {
    "NDVI": calculate_index(superX, 6, 2),
    "NDWI": calculate_index(superX, 1, 6),
    "NDBI": calculate_index(superX, 8, 6),
}


# --------------------------------------------------
# 2. Calculate difference metrics
# --------------------------------------------------

def calculate_metrics(before, after):
    before = before.detach().float().cpu().numpy()
    after = after.detach().float().cpu().numpy()

    # Exclude invalid pixels from numerical metrics
    valid = np.isfinite(before) & np.isfinite(after)
    invalid_count = int(valid.size - valid.sum())

    if valid.sum() == 0:
        return {
            "Valid pixels (%)": 0.0,
            "Invalid pixels": invalid_count,
            "Mean before": np.nan,
            "Mean after": np.nan,
            "Mean signed difference": np.nan,
            "MAE": np.nan,
            "RMSE": np.nan,
            "Difference std": np.nan,
            "Max absolute difference": np.nan,
            "Pixels |difference| > 0.05 (%)": np.nan,
        }

    b = before[valid]
    a = after[valid]
    diff = a - b

    return {
        "Valid pixels (%)": 100 * valid.mean(),
        "Invalid pixels": invalid_count,
        "Mean before": b.mean(),
        "Mean after": a.mean(),
        "Mean signed difference": diff.mean(),
        "MAE": np.abs(diff).mean(),
        "RMSE": np.sqrt(np.mean(diff ** 2)),
        "Difference std": diff.std(),
        "Max absolute difference": np.abs(diff).max(),
        "Pixels |difference| > 0.05 (%)":
            100 * np.mean(np.abs(diff) > 0.05),
    }


rows = []

for name in ["NDVI", "NDWI", "NDBI"]:
    metrics = calculate_metrics(
        indices_before[name],
        indices_after[name]
    )
    metrics["Index"] = name
    rows.append(metrics)

table = pd.DataFrame(rows).set_index("Index")

# Display a readable table
display(table.round(5))


# --------------------------------------------------
# 3. Optional: save table to Google Drive
# --------------------------------------------------

# Uncomment these lines if you want to save the CSV.
#
# from google.colab import drive
# drive.mount("/content/drive")
#
# save_path = (
#     "/content/drive/MyDrive/Project_Folder/Results/"
#     "SEN2SR_Index_Difference_Metrics.csv"
# )
# import os
# os.makedirs(os.path.dirname(save_path), exist_ok=True)
# table.to_csv(save_path)
# print("Saved:", save_path)


,Valid pixels (%),Invalid pixels,Mean before,Mean after,Mean signed difference,MAE,RMSE,Difference std,Max absolute difference,Pixels |difference| > 0.05 (%)
Index,,,,,,,,,,
NDVI,100.0,0,0.26144,0.26274,0.00130,0.01551,0.02197,0.02193,0.25958,3.82189
NDWI,100.0,0,-0.31263,-0.31349,-0.00086,0.01340,0.01909,0.01907,0.21772,2.40979
NDBI,100.0,0,0.01235,0.01386,0.00151,0.02578,0.03591,0.03588,0.37983,12.14664


## Cell 11 — Save model output to Drive as a 10-band GeoTIFF

In [36]:
import os
from google.colab import drive

# Mount Drive if it is not already mounted
drive.mount('/content/drive')

# Save location
save_dir = "/content/drive/MyDrive/Project_Folder/Results"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "SEN2SR_Before_After_Comparison1.png"
)

# Save the existing comparison figure
fig.savefig(
    save_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

print(f"Comparison saved to: {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Comparison saved to: /content/drive/MyDrive/Project_Folder/Results/SEN2SR_Before_After_Comparison1.png


## Important notes

- If the ZIP contains only visualized layers or fewer than ten raw bands, Cell 4 will stop with an explanation. Do not force the model to run on RGB-only data.
- The notebook currently assumes the 10-band SEN2SR Lite model accepts the selected input bands and scaling. Verify the checkpoint's input contract and Copernicus export scaling before interpreting scientific results.
- Difference maps compare output indices with a nearest-neighbour-resized input index. This is a diagnostic visualization, **not ground truth**.
- This notebook performs pretrained inference only. It does not fine-tune the model with a Multi-Index Spectral Loss.